Reusable function

In [1]:
import pandas as pd
import os
from pathlib import Path
from datetime import datetime
import gzip

def analyze_drive(drive_code, base_folder=r"C:\Workspace\LOCAL SANDBOX\Network_Drive_Metadata\EXPORTS"):
    """
    Analyze network drive metadata for a given drive code.
    
    Parameters:
    -----------
    drive_code : str
        Drive code prefix (e.g., 'S', 'T', 'EGIS')
    base_folder : str
        Path to the folder containing metadata files
    
    Returns:
    --------
    Prints summary statistics
    """
    
    # Find the latest file
    base_path = Path(base_folder)
    
    # Search for .csv.gz files first
    gz_files = list(base_path.glob(f"{drive_code}_network_drive_metadata_*.csv.gz"))
    csv_files = list(base_path.glob(f"{drive_code}_network_drive_metadata_*.csv"))
    
    # Combine and sort by modification time
    all_files = gz_files + csv_files
    
    if not all_files:
        print(f"❌ No metadata file found for drive code: {drive_code}")
        return
    
    latest_file = max(all_files, key=lambda x: x.stat().st_mtime)
    file_path = str(latest_file)
    
    print(f"📁 Analyzing: {os.path.basename(file_path)}\n")
    
    # Read the file and detect delimiter
    try:
        # Read first few lines to detect delimiter
        if file_path.endswith('.gz'):
            with gzip.open(file_path, 'rt', encoding='utf-8') as f:
                first_line = f.readline()
        else:
            with open(file_path, 'r', encoding='utf-8') as f:
                first_line = f.readline()
        
        # Detect delimiter
        delimiters = [',', '\t', '|', ';']
        delimiter = ','  # default
        max_count = 0
        
        for delim in delimiters:
            count = first_line.count(delim)
            if count > max_count:
                max_count = count
                delimiter = delim
        
        print(f"🔍 Detected delimiter: '{delimiter}' (found {max_count} occurrences)")
        
        # Now read the full file with detected delimiter
        if file_path.endswith('.gz'):
            df = pd.read_csv(file_path, sep=delimiter, compression='gzip', 
                           low_memory=False, quotechar='"', encoding='utf-8',
                           on_bad_lines='skip')  # Skip bad/incomplete rows
        else:
            df = pd.read_csv(file_path, sep=delimiter, low_memory=False, 
                           quotechar='"', encoding='utf-8',
                           on_bad_lines='skip')  # Skip bad/incomplete rows
        
        # Strip whitespace and quotes from column names
        df.columns = df.columns.str.strip().str.strip('"')
        
        print(f"✅ Successfully loaded {len(df):,} rows!\n")
        
        # Parse Modified column after reading if it exists
        if 'Modified' in df.columns:
            df['Modified'] = pd.to_datetime(df['Modified'], errors='coerce')
            
    except Exception as e:
        print(f"❌ Error reading file: {e}")
        print(f"\n💡 The .gz file may be corrupted. Try manually unzipping it first.")
        return
    
    # Check for required columns
    if 'Type' not in df.columns:
        print(f"❌ Error: 'Type' column not found. Available columns: {list(df.columns)}")
        return
    
    if 'SizeBytes' not in df.columns:
        print(f"❌ Error: 'SizeBytes' column not found. Available columns: {list(df.columns)}")
        return
    
    # Calculate statistics
    total_files = len(df[df['Type'] == 'File'])
    total_folders = len(df[df['Type'] == 'Folder'])
    total_bytes = df['SizeBytes'].sum()
    
    # Age distribution
    current_date = datetime.now()
    
    if 'Modified' in df.columns:
        df_with_dates = df[df['Modified'].notna()].copy()
        
        if len(df_with_dates) > 0:
            df_with_dates['age_years'] = (current_date - df_with_dates['Modified']).dt.days / 365.25
            
            less_1yr = len(df_with_dates[df_with_dates['age_years'] < 1])
            yr_1_to_5 = len(df_with_dates[(df_with_dates['age_years'] >= 1) & (df_with_dates['age_years'] < 5)])
            yr_5_to_10 = len(df_with_dates[(df_with_dates['age_years'] >= 5) & (df_with_dates['age_years'] < 10)])
            yr_10_plus = len(df_with_dates[df_with_dates['age_years'] >= 10])
        else:
            less_1yr = yr_1_to_5 = yr_5_to_10 = yr_10_plus = 0
    else:
        less_1yr = yr_1_to_5 = yr_5_to_10 = yr_10_plus = 0
        print("⚠️  Warning: 'Modified' column not found in data\n")
    
    # Top 10 extensions
    files_df = df[df['Type'] == 'File'].copy()
    
    if 'Extension' in df.columns:
        files_df['Extension_clean'] = files_df['Extension'].fillna('(no extension)').replace('', '(no extension)')
        top_extensions = files_df['Extension_clean'].value_counts().head(50)
    else:
        print("⚠️  Warning: 'Extension' column not found in data\n")
        top_extensions = pd.Series()
    
    # Print results
    print("="*70)
    print(f"DRIVE: {drive_code}")
    print("="*70)
    print(f"\n📊 BASIC COUNTS:")
    print(f"   Total Items:   {len(df):,}")
    print(f"   Total Files:   {total_files:,}")
    print(f"   Total Folders: {total_folders:,}")
    
    print(f"\n💾 TOTAL SIZE:")
    print(f"   {total_bytes / (1024**3):,.2f} GB ({total_bytes / (1024**4):,.2f} TB)")
    
    print(f"\n📅 AGE DISTRIBUTION (by Last Modified):")
    print(f"   < 1 year:    {less_1yr:,}")
    print(f"   1-5 years:   {yr_1_to_5:,}")
    print(f"   5-10 years:  {yr_5_to_10:,}")
    print(f"   10+ years:   {yr_10_plus:,}")
    
    if len(top_extensions) > 0:
        print(f"\n📄 TOP 10 FILE EXTENSIONS:")
        for i, (ext, count) in enumerate(top_extensions.items(), 1):
            print(f"   {i:2d}. {ext:20s} {count:,}")
    
    print("="*70 + "\n")

S

In [2]:
analyze_drive('S')

📁 Analyzing: S_network_drive_metadata_20260501_131009.csv.gz

🔍 Detected delimiter: '	' (found 9 occurrences)
✅ Successfully loaded 8,572,425 rows!

DRIVE: S

📊 BASIC COUNTS:
   Total Items:   8,572,425
   Total Files:   7,869,646
   Total Folders: 702,076

💾 TOTAL SIZE:
   16,405.68 GB (16.02 TB)

📅 AGE DISTRIBUTION (by Last Modified):
   < 1 year:    86,100
   1-5 years:   2,734,751
   5-10 years:  4,194,779
   10+ years:   1,556,777

📄 TOP 10 FILE EXTENSIONS:
    1. .jpg                 2,747,556
    2. .png                 1,682,233
    3. .kml                 433,243
    4. .3mxb                317,643
    5. .json                279,300
    6. .JPG                 278,101
    7. .pdf                 277,556
    8. .bin                 261,787
    9. .dae                 242,159
   10. (no extension)       125,079



T

In [3]:
analyze_drive('T')

📁 Analyzing: T_network_drive_metadata_20260501_073005.csv.gz

🔍 Detected delimiter: '	' (found 9 occurrences)
✅ Successfully loaded 14,144,603 rows!

DRIVE: T

📊 BASIC COUNTS:
   Total Items:   14,144,603
   Total Files:   13,517,635
   Total Folders: 624,144

💾 TOTAL SIZE:
   48,502.94 GB (47.37 TB)

📅 AGE DISTRIBUTION (by Last Modified):
   < 1 year:    266,867
   1-5 years:   1,781,409
   5-10 years:  2,944,016
   10+ years:   9,152,303

📄 TOP 10 FILE EXTENSIONS:
    1. .jpg                 1,262,775
    2. .hru                 835,411
    3. .sol                 824,373
    4. .gw                  821,033
    5. .chm                 688,407
    6. .mgt                 685,068
    7. .sep                 669,054
    8. .sdr                 553,564
    9. .bak                 465,062
   10. .JPG                 435,521



EGIS

In [2]:
analyze_drive('EGIS')

📁 Analyzing: EGIS_network_drive_metadata_20260430_101403.csv.gz

🔍 Detected delimiter: '	' (found 9 occurrences)
✅ Successfully loaded 40,920,029 rows!

DRIVE: EGIS

📊 BASIC COUNTS:
   Total Items:   40,920,029
   Total Files:   40,070,865
   Total Folders: 821,419

💾 TOTAL SIZE:
   167,362.85 GB (163.44 TB)

📅 AGE DISTRIBUTION (by Last Modified):
   < 1 year:    6,527,977
   1-5 years:   19,356,205
   5-10 years:  9,496,255
   10+ years:   5,539,592

📄 TOP 10 FILE EXTENSIONS:
    1. .png                 15,584,059
    2. .jpg                 8,567,643
    3. .xml                 3,150,495
    4. .jp2                 2,879,435
    5. .tif                 971,291
    6. .3mxb                824,785
    7. .tfw                 712,357
    8. .adf                 589,032
    9. .kml                 493,862
   10. (no extension)       321,238
   11. .msk                 316,633
   12. .RAW                 298,952
   13. .dae                 279,279
   14. .JPG                 264,821
   15

In [8]:
# Load the data and search for "acronym"
import pandas as pd
import gzip
from pathlib import Path

# Configuration
drive_code = 'T'  # Change this to your drive code: 'S', 'T', 'EGIS', etc.
base_folder = r"C:\Workspace\LOCAL SANDBOX\Network_Drive_Metadata\EXPORTS"
search_term = "acronym"

# Find the latest file
base_path = Path(base_folder)
gz_files = list(base_path.glob(f"{drive_code}_network_drive_metadata_*.csv.gz"))
csv_files = list(base_path.glob(f"{drive_code}_network_drive_metadata_*.csv"))
all_files = gz_files + csv_files

if not all_files:
    print(f"❌ No metadata file found for drive code: {drive_code}")
else:
    latest_file = max(all_files, key=lambda x: x.stat().st_mtime)
    file_path = str(latest_file)
    
    print(f"📁 Loading: {latest_file.name}\n")
    
    # Read the file - the delimiter is TAB (\t) not comma
    if file_path.endswith('.gz'):
        df = pd.read_csv(file_path, sep='\t', compression='gzip', low_memory=False, on_bad_lines='skip')
    else:
        df = pd.read_csv(file_path, sep='\t', low_memory=False, on_bad_lines='skip')
    
    df.columns = df.columns.str.strip().str.strip('"')
    print(f"✅ Loaded {len(df):,} rows\n")
    print(f"Available columns: {list(df.columns)}\n")
    
    # Search for the term
    print(f"🔍 Searching for '{search_term}' across all columns...\n")
    mask = pd.Series([False] * len(df))
    
    for col in df.columns:
        try:
            mask |= df[col].astype(str).str.contains(search_term, case=False, na=False, regex=False)
        except:
            pass
    
    acronym_rows = df[mask].copy()
    print(f"✅ Found {len(acronym_rows):,} rows containing '{search_term}'\n")
    print("="*70)
    
    if len(acronym_rows) > 0:
        # Set display options for better readability
        pd.set_option('display.max_rows', None)
        pd.set_option('display.max_colwidth', 80)
        pd.set_option('display.width', None)
        
        # Select key columns to display
        display_cols = ['FullPath', 'Name', 'Extension', 'Type', 'SizeBytes', 'Modified']
        available_cols = [col for col in display_cols if col in acronym_rows.columns]
        
        print(f"Displaying ALL {len(acronym_rows):,} rows:\n")
        print(acronym_rows[available_cols].to_string(index=True))
        
        print("\n" + "="*70)
        print("Columns containing the search term:")
        for col in df.columns:
            try:
                count = df[col].astype(str).str.contains(search_term, case=False, na=False, regex=False).sum()
                if count > 0:
                    print(f"   {col}: {count:,} matches")
            except:
                pass
                
        # Export to CSV for easier viewing
        output_file = f'acronym_search_results_{drive_code}.csv'
        acronym_rows.to_csv(output_file, index=False)
        print(f"\n💾 Results exported to: {output_file}")
    else:
        print("No matches found.")

📁 Loading: T_network_drive_metadata_20260501_073005.csv.gz

✅ Loaded 14,144,603 rows

Available columns: ['FullPath', 'Name', 'Type', 'Extension', 'SizeBytes', 'Created', 'Modified', 'ParentPath', 'Status', 'ErrorMessage']

🔍 Searching for 'acronym' across all columns...

✅ Found 110 rows containing 'acronym'

Displaying ALL 110 rows:

                                                                                                                                                                                                                                                                FullPath                                                                                                 Name Extension    Type  SizeBytes             Modified
76518                                                                                                                                                                                                                                  T:\EC-H\Useful 